# Retrieval Augmented Generation (RAG)

This notebook walks through **Retrieval Augmented Generation** step by step.

It covers:

1. Why LLMs hallucinate and why context windows are limited
2. Why external knowledge retrieval is needed
3. How to build a simple RAG pipeline from scratch
4. The difference between querying an LLM alone vs. using RAG

---

## Section 1 — Setup

### Why do we need RAG?

Large Language Models are powerful, but they have three fundamental limitations:

| Problem | Explanation |
|---|---|
| **Hallucination** | LLMs confidently generate plausible-sounding answers that are factually wrong, especially about topics outside their training data. |
| **Knowledge cutoff** | Training data has a fixed date. The model knows nothing about events, documents, or systems created after that date. |
| **Context window limits** | Even large context windows (128k+ tokens) cannot hold an entire knowledge base. Stuffing everything into the prompt is expensive and eventually impossible. |

**RAG solves this** by retrieving only the *relevant* pieces of external knowledge at query time and injecting them into the prompt.

### Libraries we'll use

| Library | Purpose |
|---|---|
| `openai` | Embeddings and chat completions |
| `faiss-cpu` | Fast vector similarity search |
| `numpy` | Numerical operations on embedding vectors |
| `python-dotenv` | Load `OPENAI_API_KEY` from `.env` |

### Environment: use a venv (no global installs)

Install dependencies only inside a **virtual environment**, not globally.

1. **Create and activate a venv** (in a terminal, from the project folder):
   ```bash
   python -m venv .venv
   source .venv/bin/activate   # Linux/macOS
   # or:  .venv\\Scripts\\activate   # Windows
   ```
2. **Install the packages** in that venv: `pip install -r requirements.txt` (or run the cell below)
3. **Use that venv as the notebook kernel**: In Jupyter/Cursor, choose **Kernel → Select Kernel** and pick the interpreter from `.venv` (e.g. `./.venv/bin/python`).

The notebook runs in **whatever kernel you have selected**. If the kernel is your venv, `%pip install` and all imports use that venv; no global install is needed.

In [ ]:
# Install dependencies into the current kernel's Python (use a venv kernel to avoid global install)
%pip install -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
import faiss
import numpy as np
from openai import OpenAI

# Load .env (copy env.example to .env and add your OPENAI_API_KEY)
load_dotenv()

client = OpenAI()  # reads OPENAI_API_KEY from env

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

print("Setup complete.")
print(f"Python (kernel): {sys.executable}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Chat model:      {CHAT_MODEL}")

---

## Section 2 — Querying an LLM Without RAG

Before building RAG, let's see what happens when we ask the LLM a question about **internal company knowledge** it was never trained on.

We'll create a small fictional knowledge base about an internal authentication and payments platform, then ask the LLM a question *without* giving it any of that knowledge.

### The fictional knowledge base

Imagine we work at a company called **Acme Corp** and have internal documentation about our platform.

In [ ]:
DOCUMENTS = [
    {
        "title": "Authentication Tokens",
        "content": (
            "Acme Corp uses a dual-token authentication system. When a user logs in, "
            "the auth service issues two tokens: an access token (valid for 15 minutes) "
            "and a refresh token (valid for 7 days). The access token is a signed JWT "
            "containing the user ID, role, and a session fingerprint. The refresh token "
            "is an opaque string stored server-side in Redis with a TTL of 7 days. "
            "Access tokens are sent in the Authorization header as a Bearer token. "
            "Refresh tokens are sent as an HTTP-only secure cookie."
        ),
    },
    {
        "title": "Token Refresh Flow",
        "content": (
            "When a client receives a 401 Unauthorized response, it should call "
            "POST /auth/refresh with the refresh token cookie. The auth service "
            "validates the refresh token against Redis, checks the session fingerprint, "
            "and if valid, issues a new access token and rotates the refresh token. "
            "Token rotation means the old refresh token is immediately invalidated and "
            "a new one is issued. This limits the window for token theft. If the refresh "
            "token is expired or missing from Redis, the user must re-authenticate. "
            "All refresh events are logged to the audit trail in PostgreSQL."
        ),
    },
    {
        "title": "API Rate Limiting",
        "content": (
            "Acme Corp enforces rate limiting at the API gateway level using a sliding "
            "window algorithm. Free-tier users are limited to 100 requests per minute. "
            "Pro-tier users get 1000 requests per minute. Enterprise customers have "
            "configurable limits set via their contract. Rate limit headers "
            "(X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset) are included "
            "in every response. When a client exceeds the limit, a 429 Too Many Requests "
            "response is returned with a Retry-After header. Repeated violations trigger "
            "an automatic 10-minute cooldown period."
        ),
    },
    {
        "title": "Payment Retry Policy",
        "content": (
            "Failed payment transactions in Acme Corp are retried using exponential "
            "backoff. The first retry happens after 1 hour, the second after 4 hours, "
            "the third after 24 hours. After three failed retries, the payment is marked "
            "as permanently failed and the customer is notified via email and in-app "
            "notification. The payment service uses idempotency keys to prevent duplicate "
            "charges. Each retry is logged with the failure reason (e.g., insufficient "
            "funds, card expired, processor timeout). Finance teams can view retry "
            "history in the internal dashboard under Payments > Retry Log."
        ),
    },
    {
        "title": "Service Architecture Overview",
        "content": (
            "Acme Corp runs a microservices architecture on Kubernetes. The core services "
            "are: auth-service (authentication and authorization), payment-service "
            "(billing and subscriptions), user-service (profile management), and "
            "gateway-service (API gateway, rate limiting, request routing). Services "
            "communicate via gRPC for internal calls and expose REST APIs externally. "
            "All inter-service calls require a service-to-service JWT signed with a "
            "shared secret rotated weekly. Logs are aggregated in Elasticsearch and "
            "dashboards are in Grafana."
        ),
    },
]

print(f"Knowledge base contains {len(DOCUMENTS)} documents:\n")
for doc in DOCUMENTS:
    print(f"  - {doc['title']} ({len(doc['content'])} chars)")

### Asking the LLM without any context

We include a **system prompt guardrail** that tells the model to only answer from provided context. Since we're providing no context, the model should admit it doesn't know.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant for Acme Corp employees. "
    "You must only answer using the provided context. "
    "If the context does not contain enough information to answer, "
    "say: 'I don't have enough information to answer that.'"
)

question = "How does our authentication system refresh expired tokens?"

response_no_rag = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ],
    temperature=0,
)

print("Question:")
print(f"  {question}\n")
print("LLM Response (no context provided):")
print(f"  {response_no_rag.choices[0].message.content}")

### What happened?

The LLM either:
- Admitted it doesn't have the information (good — the guardrail worked), or
- Hallucinated a generic answer about token refresh (bad — it made up details)

Either way, the model **does not know** Acme Corp's specific authentication flow.

### Why not just paste all the documents into the prompt?

With only 5 short documents, we *could* paste everything into the prompt. But consider:

- Real companies have **thousands** of documents
- Context windows have **token limits** (and cost scales with tokens)
- Stuffing irrelevant documents **dilutes** the model's attention

This is exactly why we need **RAG** — retrieve only what's relevant, then generate.

---

## Section 3 — Introducing RAG

**Retrieval Augmented Generation** adds a retrieval step before generation.

The pipeline has two phases:

### Pre-processing phase (done once, before any user queries)

```
Documents → Chunking → Embeddings → Vector Store
```

1. **Document ingestion** — load source material
2. **Chunking** — split documents into smaller pieces
3. **Embedding** — convert each chunk into a numerical vector
4. **Indexing** — store vectors in a searchable index

### Query-time phase (when a user asks a question)

```
Query → Embedding → Vector Search → Retrieved Chunks → Augmented Prompt → LLM → Response
```

1. **Query embedding** — convert the user's question into a vector
2. **Retrieval** — find the most similar chunks from the vector store
3. **Augmentation** — inject retrieved chunks into the prompt
4. **Generation** — the LLM generates a response grounded in retrieved context

We'll build each step now.

---

## Section 4 — Document Ingestion

In a real system, documents come from many sources: PDFs, documentation sites, code repositories, wikis, databases, etc.

The documents are defined in Python variables above. Here we load them into a list and print them.

In [ ]:
raw_texts = [doc["content"] for doc in DOCUMENTS]

print(f"Loaded {len(raw_texts)} documents\n")
print("=" * 60)
for i, (doc, text) in enumerate(zip(DOCUMENTS, raw_texts)):
    print(f"\nDocument {i + 1}: {doc['title']}")
    print("-" * 40)
    print(text)
    print()

---

## Section 5 — Chunking

### Why chunk?

Documents are often too long to embed as a single unit. Large chunks:
- Mix multiple topics, making similarity search less precise
- Use more tokens per retrieval hit

Chunks that are too small lose context and become meaningless fragments.

### Strategy: fixed-size chunking with overlap

This is the simplest and most common approach:
- **chunk_size** — maximum number of characters per chunk
- **chunk_overlap** — characters shared between consecutive chunks (prevents losing context at boundaries)

```
Chunk 1: chars 0–400
Chunk 2: chars 350–750   (50 char overlap)
Chunk 3: chars 700–1100
```

In [ ]:
def chunk_text(text: str, chunk_size: int = 400, chunk_overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks of roughly chunk_size characters."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - chunk_overlap
    return chunks


# TODO: Try changing chunk_size (e.g., 200, 600) and chunk_overlap (e.g., 0, 100)
#       and observe how the number and content of chunks change.

all_chunks: list[str] = []
chunk_metadata: list[dict] = []

for doc in DOCUMENTS:
    doc_chunks = chunk_text(doc["content"])
    for chunk in doc_chunks:
        all_chunks.append(chunk)
        chunk_metadata.append({"title": doc["title"]})

print(f"Total chunks created: {len(all_chunks)}\n")
for i, (chunk, meta) in enumerate(zip(all_chunks, chunk_metadata)):
    print(f"Chunk {i} (from '{meta['title']}'):")
    print(f"  Length: {len(chunk)} chars")
    print(f"  Preview: {chunk[:120]}...")
    print()

---

## Section 6 — Generating Embeddings

### What are embeddings?

An embedding converts text into a **dense numerical vector** (a list of floating-point numbers). Texts with similar meanings end up with vectors that are close together in this high-dimensional space.

This is what enables **semantic search** — we don't match keywords, we match *meaning*.

```
"How do I refresh a token?" → [0.12, -0.45, 0.78, ...]
"Token renewal process"     → [0.11, -0.43, 0.80, ...]   ← very similar!
"Payment retry policy"      → [-0.32, 0.67, 0.01, ...]   ← very different
```

We'll use OpenAI's `text-embedding-3-small` model, which produces 1536-dimensional vectors.

In [ ]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Generate embeddings for a list of texts using OpenAI."""
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


print("Generating embeddings for all chunks...")
chunk_embeddings = get_embeddings(all_chunks)
print(f"Done. Generated {len(chunk_embeddings)} embeddings.\n")

sample = chunk_embeddings[0]
print(f"Each embedding has {len(sample)} dimensions.")
print(f"First 10 values of chunk 0's embedding:")
print(f"  {sample[:10]}")

Each chunk is now represented as a 1536-dimensional vector. These vectors capture the **semantic meaning** of the text — chunks about similar topics will have vectors pointing in similar directions.

---

## Section 7 — Creating the Vector Index

We need a way to **efficiently search** through our embedding vectors. This is what a vector store (or vector index) does.

**FAISS** (Facebook AI Similarity Search) is a library for fast nearest-neighbor search over dense vectors. For our small dataset we'll use a flat (brute-force) index — in production you'd use approximate methods for speed.

In [ ]:
embedding_dim = len(chunk_embeddings[0])
embeddings_array = np.array(chunk_embeddings, dtype="float32")

# Normalize vectors so that inner product = cosine similarity (OpenAI embeddings are cosine-optimized)
faiss.normalize_L2(embeddings_array)

index = faiss.IndexFlatIP(embedding_dim)  # Inner product on normalized vectors = cosine similarity
index.add(embeddings_array)

print("FAISS index created.")
print(f"  Vector dimensionality: {embedding_dim}")
print(f"  Vectors in index:      {index.ntotal}")

We also keep our `all_chunks` and `chunk_metadata` lists around — FAISS stores vectors only, so we need the mapping from index position back to the original chunk text.

---

## Section 8 — Query Embedding and Retrieval

Now the exciting part. We take a user question, embed it, and search the vector index for the most similar chunks.

The idea: if the question's embedding is close to a chunk's embedding, that chunk probably contains relevant information.

In [ ]:
# TODO: Try different queries and see which chunks get retrieved.
#       Examples:
#         "What happens when a payment fails?"
#         "How are services connected?"
#         "What are the rate limits for free users?"

query = "How does token refresh work?"
top_k = 3

print(f"Query: '{query}'")
print(f"Retrieving top {top_k} most similar chunks...\n")

query_embedding = get_embeddings([query])[0]
query_vector = np.array([query_embedding], dtype="float32")
faiss.normalize_L2(query_vector)  # match index: inner product = cosine similarity

scores, indices = index.search(query_vector, top_k)  # IndexFlatIP returns similarity (higher = better)

retrieved_chunks: list[str] = []
for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
    chunk_text_result = all_chunks[idx]
    meta = chunk_metadata[idx]
    retrieved_chunks.append(chunk_text_result)
    print(f"Rank {rank + 1} | Similarity: {score:.4f} | Source: {meta['title']}")
    print(f"  {chunk_text_result[:200]}...")
    print()

### Reading the results

- **Higher similarity = more similar** (we use cosine similarity via L2-normalized inner product, which matches OpenAI's recommendation for `text-embedding-3-small`)
- The top chunks should be from the "Token Refresh Flow" and "Authentication Tokens" documents
- These chunks likely contain the answer to our question

---

## Section 9 — Prompt Augmentation

We now **augment** the prompt by injecting the retrieved chunks as context. This is the "A" in RAG.

The prompt structure:

```
System: Instructions + guardrails
Context: [Retrieved chunk 1]
         [Retrieved chunk 2]
         [Retrieved chunk 3]
User: The original question
```

The system prompt tells the LLM to ground its answer in the provided context.

In [ ]:
def build_augmented_prompt(system: str, context_chunks: list[str], user_query: str) -> list[dict]:
    """Build a chat messages list with retrieved context injected."""
    context_block = "\n\n---\n\n".join(
        f"[Context {i + 1}]\n{chunk}" for i, chunk in enumerate(context_chunks)
    )

    system_with_context = (
        f"{system}\n\n"
        f"Use the following context to answer the user's question:\n\n"
        f"{context_block}"
    )

    return [
        {"role": "system", "content": system_with_context},
        {"role": "user", "content": user_query},
    ]


augmented_messages = build_augmented_prompt(SYSTEM_PROMPT, retrieved_chunks, question)

print("=" * 60)
print("AUGMENTED PROMPT")
print("=" * 60)
for msg in augmented_messages:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"])
print("\n" + "=" * 60)

Notice how the prompt now contains the **specific internal documentation** that's relevant to the question. The LLM no longer has to guess — it has the actual source material.

---

## Section 10 — Generation With RAG

Let's send the augmented prompt to the LLM and compare the result with the earlier response (without RAG).

In [ ]:
response_with_rag = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=augmented_messages,
    temperature=0,
)

print("Question:")
print(f"  {question}\n")

print("=" * 60)
print("RESPONSE WITHOUT RAG")
print("=" * 60)
print(response_no_rag.choices[0].message.content)

print("\n" + "=" * 60)
print("RESPONSE WITH RAG")
print("=" * 60)
print(response_with_rag.choices[0].message.content)

### What changed?

The RAG response should now include **specific details** from Acme Corp's documentation:

- The `POST /auth/refresh` endpoint
- Redis-backed refresh tokens
- Token rotation (old token invalidated, new one issued)
- Session fingerprint validation
- Audit trail logging in PostgreSQL

None of this information exists in the LLM's training data. It came from our retrieved context.

**That's RAG in action.** The model is *generating* an answer, but it's *grounded* in *retrieved* information.

---

## Section 11 — Experimentation

Things to try:

### 1. Change the chunk size

Go back to Section 5 and try:
- `chunk_size=200` — smaller chunks, more of them, more precise but less context
- `chunk_size=800` — larger chunks, fewer of them, more context but less precise

Re-run from Section 5 onwards and observe how retrieval quality changes.

### 2. Change `top_k`

In Section 8, try:
- `top_k=1` — only the single best match
- `top_k=5` — more context, but possibly less relevant chunks mixed in

### 3. Modify the system prompt

Try removing the guardrail ("if the context doesn't have the answer, say so") and see if the model starts hallucinating details even with RAG.

### 4. Ask different questions

Try questions that span multiple documents:
- "What happens if a payment fails and the customer tries to authenticate?"
- "What rate limits apply to the auth service?"
- "How do internal services communicate?"

### 5. Add a new document

Add a 6th document to the `DOCUMENTS` list (e.g., about logging or deployment). Re-run the pipeline and ask a question about the new topic.

### Key parameters that affect RAG behavior

| Parameter | Effect |
|---|---|
| `chunk_size` | Larger = more context per chunk, less precision. Smaller = more precision, less context. |
| `chunk_overlap` | Higher overlap = less information lost at boundaries, but more total chunks. |
| `top_k` | More retrieved chunks = more context for the LLM, but may include irrelevant material. |
| `temperature` | Higher = more creative responses. Lower = more deterministic, factual. |
| System prompt | Controls whether the model sticks to context or falls back to training data. |

In [ ]:
# TODO: Use this cell as a sandbox. Try a full RAG query with different parameters.

# --- Experiment settings ---
experiment_query = "What happens when a payment fails?"  # TODO: change this
experiment_top_k = 3  # TODO: try 1, 2, 5

# Embed and retrieve
q_emb = get_embeddings([experiment_query])[0]
q_vec = np.array([q_emb], dtype="float32")
dists, idxs = index.search(q_vec, experiment_top_k)

exp_chunks = [all_chunks[i] for i in idxs[0]]

print(f"Query: '{experiment_query}'\n")
print("Retrieved chunks:")
for rank, (d, i) in enumerate(zip(dists[0], idxs[0])):
    print(f"  {rank + 1}. [{chunk_metadata[i]['title']}] dist={d:.4f}")

# Build prompt and generate
exp_messages = build_augmented_prompt(SYSTEM_PROMPT, exp_chunks, experiment_query)
exp_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=exp_messages,
    temperature=0,
)

print(f"\nAnswer:\n{exp_response.choices[0].message.content}")